# Gold `fact_province_year`

One row per reporting unit and year (100 × 6 = 600). It brings tourism, economy and population together so the views can compare them.

| | |
| --- | --- |
| **Key** | (`province_psgc`, `year`) |
| **Foreign keys** | `dim_province.province_psgc`, `dim_year.year` |
| **Built from** | `dot_travelers_unit_year`, `psa_economy_unit_year`, `population_unit_year` |
| **Requires** | `dim_province` and `dim_year` already built |

**Rules**

* The grain is every unit × every year. A year a source does not cover is NULL, never 0 (DL-009).
* Columns are copied from Silver; nothing is recomputed. Ratios, shares and growth belong to the views.
* `comparable_to_2019` is year-specific and comes from DOT (DL-013, DL-020).
* Units are written as column comments from `ref_indicator` (D17).

## Configuration and helpers

`dq(...)` records each check. `finish(...)` saves the results, stops on any `BLOCK` failure, then writes the table and closes the run log.

In [0]:
from pyspark.sql import functions as F
import uuid
from datetime import datetime

CATALOG = "tuklas_dev"
SILVER_SCHEMA = "03_silver"
GOLD_SCHEMA = "04_gold"
DQ_SCHEMA = "06_data_quality"

NOTEBOOK_NAME = "01_gold_dim_province"      # 02_gold_dim_year / 03_gold_fact_province_year
run_id = f"gold_{NOTEBOOK_NAME[3:]}_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

def silver_table(n): return f"`{CATALOG}`.`{SILVER_SCHEMA}`.`{n}`"
def gold_table(n):   return f"`{CATALOG}`.`{GOLD_SCHEMA}`.`{n}`"
def dq_table(n):     return f"`{CATALOG}`.`{DQ_SCHEMA}`.`{n}`"

dq_results = []

def dq(check_id, table_name, severity, failed_rows, expected, observed, sample_keys=None, notes=""):
    """Record one data-quality check result. failed_rows = 0 means PASS."""
    status = "PASS" if failed_rows == 0 else "FAIL"
    keys = " | ".join(str(k) for k in list(sample_keys)[:20]) if sample_keys else None
    dq_results.append((run_id, NOTEBOOK_NAME, check_id, table_name, severity, status, int(failed_rows),
                       str(expected), str(observed), keys, notes))
    marker = "✓" if status == "PASS" else ("✗" if severity == "BLOCK" else "!")
    print(f"{marker} {check_id:<11} {severity:<6} {status:<4} | {table_name} | expected {expected} | observed {observed}"
          + (f" | e.g. {keys}" if keys else ""))

def sample(df, *cols, n=20):
    return [" / ".join("" if r[c] is None else str(r[c]) for c in cols) for r in df.select(*cols).limit(n).collect()]

spark.sql(f"""
INSERT INTO {gold_table('gold_run_log')}
(run_id, notebook_name, started_at, completed_at, tables_written, rows_written,
 dq_checks_run, dq_block_failures, dq_review_flags, status, notes)
VALUES (:run_id, :nb, current_timestamp(), NULL, 0, 0, 0, 0, 0, 'STARTED', 'Notebook execution started')
""", args={"run_id": run_id, "nb": NOTEBOOK_NAME})
print(f"Run ID: {run_id}")

def finish(tables):
    """Save DQ results; stop on any BLOCK failure (nothing written); otherwise overwrite each Gold table and close the run log."""
    dq_df = (spark.createDataFrame(dq_results,
        "run_id STRING, notebook_name STRING, check_id STRING, table_name STRING, severity STRING, status STRING, "
        "failed_rows BIGINT, expected STRING, observed STRING, sample_keys STRING, notes STRING")
        .withColumn("checked_at", F.current_timestamp()))
    display(dq_df.select("check_id", "severity", "status", "failed_rows", "expected", "observed", "sample_keys"))
    dq_df.select("run_id", "notebook_name", "check_id", "table_name", "severity", "status", "failed_rows",
                 "expected", "observed", "sample_keys", "checked_at", "notes") \
         .write.format("delta").mode("append").saveAsTable(dq_table("dq_results"))

    blocks = [r for r in dq_results if r[4] == "BLOCK" and r[5] == "FAIL"]
    reviews = [r for r in dq_results if r[4] == "REVIEW" and r[5] == "FAIL"]

    def update(status, tw, rw, notes):
        spark.sql(f"""
        UPDATE {gold_table('gold_run_log')}
        SET status = :status, completed_at = current_timestamp(), tables_written = :tw, rows_written = :rw,
            dq_checks_run = :dc, dq_block_failures = :db, dq_review_flags = :dr, notes = :notes
        WHERE run_id = :run_id
        """, args={"status": status, "tw": tw, "rw": rw, "dc": len(dq_results),
                   "db": len(blocks), "dr": len(reviews), "notes": notes, "run_id": run_id})

    if blocks:
        msg = "BLOCK checks failed: " + ", ".join(r[2] for r in blocks) + ". No Gold tables written."
        update("FAILED", 0, 0, msg)
        raise AssertionError(msg + f" See {dq_table('dq_results')} for run_id = '{run_id}'.")

    written = {}
    for name, df in tables.items():
        (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_table(name)))
        written[name] = spark.table(gold_table(name)).count()
        print(f"✓ {gold_table(name)}: {written[name]} rows")

    status = "SUCCESS_WITH_REVIEW" if reviews else "SUCCESS"
    notes = (f"{', '.join(f'{k}={v}' for k, v in written.items())}; {len(dq_results)} checks, {len(reviews)} for review"
             + (f" ({', '.join(r[2] for r in reviews)})" if reviews else "") + ".")
    update(status, len(written), sum(written.values()), notes)
    print(f"Run {run_id}: {status}\n{notes}")

## Step 1: Build

The grid is `dim_province` × `dim_year`. Each Silver table is joined on (`unit_psgc_code`, `year`). Silver already has one row per unit and year, so the joins cannot add rows.

In [0]:
from functools import reduce

EXPECTED_UNITS, YEARS, BASELINE_YEAR = 100, [2019, 2020, 2021, 2022, 2023, 2024], 2019
VALID_STATUS = {"reported", "dash_as_zero", "reported_partial", "not_reported"}   # DL-019
EXPECTED_UNITS_WITHOUT_DOT = {"0906600000", "1900700000", "1903600000", "1907000000", "1908700000", "1908800000"}

dim_province = spark.table(gold_table("dim_province"))
dim_year = spark.table(gold_table("dim_year"))
dot = spark.table(silver_table("dot_travelers_unit_year"))
econ = spark.table(silver_table("psa_economy_unit_year"))
pop = spark.table(silver_table("population_unit_year"))
ref = spark.table(silver_table("ref_indicator"))

tour_cols = ["foreign_travelers", "overseas_filipinos", "domestic_travelers", "total_travelers"]
econ_cols = ["gdp_total_current", "gdp_total_constant", "gdp_per_capita_current", "afs_gva_current", "afs_gva_constant"]

grid = (dim_province.select(F.col("province_psgc").alias("unit_psgc_code")).crossJoin(dim_year.select("year")))
d = dot.select("unit_psgc_code", "year", *tour_cols, "traveler_value_status", "comparable_to_2019")
e = econ.select("unit_psgc_code", "year", *econ_cols)
p = pop.select("unit_psgc_code", "year", "population_total", "population_source")

fact = (grid.join(d, ["unit_psgc_code", "year"], "left")
            .join(e, ["unit_psgc_code", "year"], "left")
            .join(p, ["unit_psgc_code", "year"], "left")
            .withColumnRenamed("unit_psgc_code", "province_psgc")
            .withColumn("built_at", F.current_timestamp())
            .select("province_psgc", "year", *tour_cols, "traveler_value_status", *econ_cols,
                    "population_total", "population_source", "comparable_to_2019", "built_at"))
display(fact.groupBy("year").agg(F.count("*").alias("rows"), F.count("total_travelers").alias("with_travelers"),
        F.count("gdp_total_current").alias("with_gdp"), F.count("population_total").alias("with_population")).orderBy("year"))

## Step 2: Data-quality checks

| ID | Check | Severity |
| --- | --- | --- |
| DQ-GLD-03 | 600 rows, one per (province, year) | BLOCK |
| DQ-GLD-04 | Every key exists in the dimensions | BLOCK |
| DQ-GLD-05 | Tourism columns equal Silver DOT | BLOCK |
| DQ-GLD-06 | Economy columns equal Silver PSA | BLOCK |
| DQ-GLD-07 | Population equals Silver population | BLOCK |
| DQ-GLD-08 | `traveler_value_status` uses the DL-019 values; `not_reported` rows have NULL travelers | BLOCK |
| DQ-GLD-09 | `total_travelers` ≥ each part | REVIEW |
| DQ-GLD-10 | A&F GVA ≤ GDP (current and constant) | BLOCK |
| DQ-GLD-11 | Units without DOT rows stay NULL, not 0 | BLOCK |
| DQ-GLD-12 | A source's columns are NULL outside its years | BLOCK |
| DQ-GLD-13 | `comparable_to_2019` is true for every unit in 2019 | BLOCK |
| DQ-GLD-14 | Every `ref_indicator` measure is a fact column | BLOCK |

In [0]:
yr = lambda df: sorted({r[0] for r in df.select("year").distinct().collect()})
dot_years, econ_years, pop_years = yr(dot), yr(econ), yr(pop)

# grain
n = fact.count()
dup = fact.groupBy("province_psgc", "year").count().filter("count > 1")
dq("DQ-GLD-03", "fact_province_year", "BLOCK", dup.count() + (0 if n == EXPECTED_UNITS * len(YEARS) else 1),
   f"{EXPECTED_UNITS} x {len(YEARS)} = {EXPECTED_UNITS * len(YEARS)} rows, 0 duplicates",
   f"{n} rows, {dup.count()} duplicates", sample(dup, "province_psgc", "year"))

# keys exist in the dimensions
orph_p = fact.join(dim_province.select("province_psgc"), "province_psgc", "left_anti")
orph_y = fact.join(dim_year.select("year"), "year", "left_anti")
dq("DQ-GLD-04", "fact_province_year", "BLOCK", orph_p.count() + orph_y.count(), "every key in the dimensions",
   f"{orph_p.count()} unknown provinces, {orph_y.count()} unknown years")

# fact equals Silver
def total(df, c): return df.agg(F.sum(F.col(c).cast("decimal(38,0)"))).first()[0] or 0
diff = [c for c in tour_cols if total(fact, c) != total(dot, c)]
rows_ok = fact.filter("traveler_value_status is not null").count() == dot.count()
dq("DQ-GLD-05", "fact_province_year", "BLOCK", len(diff) + (0 if rows_ok else 1),
   "tourism sums and row count equal Silver DOT", f"differs: {diff}" if diff else "equal")
diff = [c for c in econ_cols if total(fact, c) != total(econ, c)]
dq("DQ-GLD-06", "fact_province_year", "BLOCK", len(diff), "economy sums equal Silver PSA", f"differs: {diff}" if diff else "equal")
diff = [] if total(fact, "population_total") == total(pop, "population_total") else ["population_total"]
dq("DQ-GLD-07", "fact_province_year", "BLOCK", len(diff), "population sum equals Silver population", f"differs: {diff}" if diff else "equal")

# status values, and not_reported has no numbers (DL-019)
bad_status = fact.filter(F.col("traveler_value_status").isNotNull() & ~F.col("traveler_value_status").isin(*VALID_STATUS))
nr_value = fact.filter((F.col("traveler_value_status") == "not_reported") & F.col("total_travelers").isNotNull())
dq("DQ-GLD-08", "fact_province_year", "BLOCK", bad_status.count() + nr_value.count(),
   "DL-019 statuses; not_reported rows have NULL travelers",
   f"{bad_status.count()} bad statuses, {nr_value.count()} not_reported rows with a value",
   sample(bad_status.unionByName(nr_value), "province_psgc", "year", "traveler_value_status"))

# total vs parts
parts = fact.filter((F.col("total_travelers") < F.col("domestic_travelers")) | (F.col("total_travelers") < F.col("foreign_travelers")))
dq("DQ-GLD-09", "fact_province_year", "REVIEW", parts.count(), "total >= each part", f"{parts.count()} violations",
   sample(parts, "province_psgc", "year"))

# A&F GVA <= GDP
over = fact.filter((F.col("afs_gva_current") > F.col("gdp_total_current")) | (F.col("afs_gva_constant") > F.col("gdp_total_constant")))
dq("DQ-GLD-10", "fact_province_year", "BLOCK", over.count(), "A&F GVA <= GDP", f"{over.count()} violations", sample(over, "province_psgc", "year"))

# missing is never 0 (DL-009)
bad_missing = fact.filter(F.col("province_psgc").isin(*EXPECTED_UNITS_WITHOUT_DOT)
                          & (F.col("total_travelers").isNotNull() | (F.col("traveler_value_status") != "not_reported")))
dq("DQ-GLD-11", "fact_province_year", "BLOCK", bad_missing.count(), "units without DOT rows are NULL / not_reported",
   f"{bad_missing.count()} rows with a value", sample(bad_missing, "province_psgc", "year"))

# years outside a source's range are NULL
def outside(cols, years):
    return fact.filter(~F.col("year").isin(*years) & reduce(lambda a, b: a | b, [F.col(c).isNotNull() for c in cols])).count()
n_out = outside(econ_cols, econ_years) + outside(["population_total"], pop_years) + outside(tour_cols, dot_years)
dq("DQ-GLD-12", "fact_province_year", "BLOCK", n_out, "a source's columns are NULL outside its years", f"{n_out} values outside range")

# baseline year is comparable with itself
bad_base = fact.filter((F.col("year") == BASELINE_YEAR) & (F.col("comparable_to_2019") != True))
dq("DQ-GLD-13", "fact_province_year", "BLOCK", bad_base.count(), f"comparable_to_2019 = true in {BASELINE_YEAR}",
   f"{bad_base.count()} units", sample(bad_base, "province_psgc"))

# every indicator in ref_indicator is a fact column
missing_cols = [r[0] for r in ref.select("measure").collect() if r[0] not in fact.columns]
dq("DQ-GLD-14", "fact_province_year", "BLOCK", len(missing_cols), "every ref_indicator measure is a fact column",
   f"missing: {missing_cols}" if missing_cols else "all present", missing_cols)

## Step 3: Write and add column comments

Units come from `ref_indicator`, so they cannot drift from Silver.

In [0]:
finish({"fact_province_year": fact})

# Column comments (D17). Units come from ref_indicator so they cannot drift from Silver.
FACT = gold_table("fact_province_year")
for r in ref.collect():
    note = f"{r['title']} | unit: {r['unit']} | {r['price_basis']} | {r['additivity']}".replace("'", "''")
    spark.sql(f"ALTER TABLE {FACT} ALTER COLUMN `{r['measure']}` COMMENT '{note}'")
for c in tour_cols:
    spark.sql(f"ALTER TABLE {FACT} ALTER COLUMN `{c}` COMMENT 'DOT traveler count, as reported (persons)'")
spark.sql(f"ALTER TABLE {FACT} ALTER COLUMN `population_total` COMMENT 'Population, persons; source in population_source'")
spark.sql(f"ALTER TABLE {FACT} ALTER COLUMN `traveler_value_status` COMMENT 'reported | dash_as_zero | reported_partial | not_reported (DL-019)'")
spark.sql(f"ALTER TABLE {FACT} ALTER COLUMN `comparable_to_2019` COMMENT 'true if DOT counts this unit the same way as in 2019 (DL-013, DL-020)'")
print("✓ Column comments added.")